# Improved Fix Metadata Demo

This notebook demonstrates how to use the Curator workflows client library to fix invalid values in tabular data. It follows the same patterns as the extract_metadata demo, using the FixMetadataClient class to handle the workflow.

## Setup and Configuration

In [ ]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv

# Import the client library
sys.path.append('../../src')
from fix_client import FixMetadataClient

# Load environment variables
load_dotenv()

# Create client
client = FixMetadataClient()

print(f"API URL: {client.api_url}")
print(f"API Key configured: {'Yes' if client.api_key else 'No'}")

## 1. Fix Metadata from a Local File

### 1.1 Define File Paths

In [ ]:
# Define the file path and ground truth path
file_path = "data/fix_metadata_source.csv"
ground_truth_path = "data/fix_metadata_source_gt.csv"

# Check if files exist
if not os.path.exists(file_path):
    print(f"❌ File not found at: {file_path}")
else:
    print(f"📄 Found file: {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")

if not os.path.exists(ground_truth_path):
    print(f"❌ Ground truth file not found at: {ground_truth_path}")
else:
    print(f"📄 Found ground truth file: {ground_truth_path}")

### 1.2 Fix Metadata Using the Client

In [ ]:
print("🔧 Fixing metadata in CSV file...")

# Define a custom callback function to display status updates
def fix_status_callback(status):
    """Display status updates during the fix metadata process."""
    print(f"Status: {status['status']}")
    if 'message' in status:
        print(f"Message: {status['message']}")
    print("---")

# Use the fix_metadata method from the FixMetadataClient
result = client.fix_metadata(
    file_path=file_path,
    schema_name="nf",
    output_path="fixed_metadata.csv",
    callback=fix_status_callback
)

### 1.3 Display the Results

In [ ]:
print("\n📊 Fix Metadata Results:")
print(f"Status: {result['status']}")

if 'output_file' in result:
    print(f"Output file: {result['output_file']}")
    
    # Display the first few rows of the fixed data
    if 'data' in result:
        print("\nFirst few rows of fixed data:")
        display(result['data'].head())
    
    # Display correction statistics
    if 'corrections' in result:
        print("\n📈 Correction Statistics:")
        print(f"Total corrections made: {result['correction_count']}")
        
        print("\nCorrections by column:")
        for column, details in result['corrections'].items():
            print(f"\nColumn: {column}")
            print(f"Number of corrections: {details['count']}")
            
            if details['examples']:
                print("Examples of corrections:")
                for example in details['examples']:
                    print(f"  Row {example['row']}: '{example['original']}' → '{example['corrected']}'")

### 1.4 Compare with Ground Truth

In [ ]:
print("\n🔍 Comparing with Ground Truth:")

try:
    # Read ground truth and fixed data
    df_ground_truth = pd.read_csv(ground_truth_path)
    df_fixed = pd.read_csv(result['output_file'])
    
    # Ensure both dataframes have the same index
    if 'id' in df_ground_truth.columns and 'id' in df_fixed.columns:
        df_ground_truth = df_ground_truth.set_index('id')
        df_fixed = df_fixed.set_index('id')
    
    # Find differences between fixed and ground truth
    differences = {}
    for column in df_ground_truth.columns:
        if column in df_fixed.columns:
            mask = df_fixed[column] != df_ground_truth[column]
            if mask.any():
                differences[column] = {
                    'row_id': mask[mask].index.tolist(),
                    'fixed_values': df_fixed[column][mask].tolist(),
                    'ground_truth_values': df_ground_truth[column][mask].tolist()
                }
    
    # Display differences
    if differences:
        print("\nFound differences in the following columns:")
        for col, diff in differences.items():
            print(f"\nColumn: {col}")
            for i in range(len(diff['row_id'])):
                print(f"Row ID: {diff['row_id'][i]}")
                print(f"Fixed: {diff['fixed_values'][i]}")
                print(f"Ground Truth: {diff['ground_truth_values'][i]}")
    else:
        print("\n✅ No differences found between fixed data and ground truth!")
    
    # Calculate accuracy metrics
    total_cells = len(df_ground_truth.index) * len(df_ground_truth.columns)
    incorrect_cells = sum(len(diff['row_id']) for diff in differences.values())
    accuracy = (total_cells - incorrect_cells) / total_cells * 100
    
    print(f"\n📊 Overall Accuracy: {accuracy:.2f}%")
    
except Exception as e:
    print(f"❌ Failed to compare with ground truth: {str(e)}")

## 3. Batch Processing Multiple Files

For batch processing of multiple files, you could implement a similar approach to the extract_metadata demo:

In [ ]:
# Example code for batch processing
def process_multiple_files(file_paths, schema_name, output_dir=None):
    """Process multiple files in sequence."""
    results = []
    
    for file_path in file_paths:
        print(f"Processing {file_path}...")
        result = client.fix_metadata(
            file_path=file_path,
            schema_name=schema_name,
            output_path=os.path.join(output_dir or "", f"fixed_{os.path.basename(file_path)}") if output_dir else None,
            callback=fix_status_callback
        )
        results.append({
            "file_path": file_path,
            "status": result["status"],
            "output_file": result.get("output_file"),
            "correction_count": result.get("correction_count", 0)
        })
    
    return results

# Example usage (commented out)
'''
# Define a list of files to process
files_to_process = [
    "data/file1.csv",
    "data/file2.csv",
    "data/file3.csv"
]

# Process all files
batch_results = process_multiple_files(
    file_paths=files_to_process,
    schema_name="nf",
    output_dir="output"
)

# Display batch results
for result in batch_results:
    print(f"File: {result['file_path']}")
    print(f"Status: {result['status']}")
    print(f"Output: {result['output_file']}")
    print(f"Corrections: {result['correction_count']}")
    print("---")
'''